# 11.10 - Human-in-the-Loop

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook builds every pattern that decides when an agent may act alone and when a human must step in: an approval gate on irreversible actions, a confidence router on uncertain ones, the durable interrupt/resume primitive and its re-run trap, tiered escalation with context, the escalation-fatigue curve, an earned-and-revocable trust ladder, and a framework picker. Almost all of it runs with no API key, because oversight is decision logic, not a model call.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - dependencies

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

This lesson is deliberately keyless: the only third-party package is python-dotenv, and even that is optional. The line is commented so it no-ops on a machine that already has it; uncomment on a fresh Colab runtime.

**How the code works - step by step**
- **`# !pip install python-dotenv -q`** - the sole dependency, commented out because most of the notebook needs nothing beyond the standard library.

**In one line:** optional env-file loader, nothing else to install.

**What you'll see:** (no output - environment prep)

## Setup - environment keys

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Reads provider keys from the environment rather than hardcoding them. Almost every cell in this lesson runs without any key at all; these slots only matter if you extend the demos into real model calls.

**How the code works - step by step**
- **`import os`** - standard-library access to environment variables.
- **`os.environ.setdefault(...)` x3** - seeds `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GOOGLE_API_KEY` to empty strings only if they are not already set, so nothing is overwritten and no key is ever pasted into the file.

**In one line:** pull keys from the environment, never the source.

**What you'll see:** (no output - environment prep)

## 1 - Build an approval gate

The foundational move every later pattern refines: before the agent executes anything, classify the action by risk. Read-only actions auto-run; irreversible ones - spend money, send a message, delete a record - pause for a human. The set of risky actions is not a detail, it IS the safety boundary, so it is written as plain data. Runs keyless and deterministic.

In [ ]:
# APPROVAL GATES - the risk boundary. Read-only actions auto-execute; actions that
# spend money, send messages, or delete data PAUSE for a human to approve. The list of
# risky actions IS the safety mechanism. We script the "agent" so this runs with no key.

REQUIRES_APPROVAL = {"send_email", "process_refund", "delete", "purchase", "book_course"}

def gate(action):
    if action["tool"] in REQUIRES_APPROVAL:
        return "PAUSE (needs approval)"     # irreversible -> a human must sign off
    return "AUTO (read-only, safe)"         # read-only -> execute immediately

# The agent proposes these actions (tool + args):
proposed = [
    {"tool": "search",         "args": {"q": "course price"}},
    {"tool": "process_refund", "args": {"order": "12345", "amount": 50000}},
    {"tool": "send_email",     "args": {"to": "student@test.com"}},
    {"tool": "delete",         "args": {"record": "user-9"}},
]
print("Every proposed action passes the approval gate first:")
pending = []
for a in proposed:
    verdict = gate(a)
    print(f"  {a['tool']:14} -> {verdict}")
    if verdict.startswith("PAUSE"):
        pending.append(a)

# The human resolves the paused actions:
print(f"\n{len(pending)} actions paused for a human.")
print("  human APPROVES process_refund -> executed")
print("  human REJECTS  delete          -> discarded")
print("Read-only actions never waited; only the irreversible ones needed a person.")
# Output:
# Every proposed action passes the approval gate first:
#   search         -> AUTO (read-only, safe)
#   process_refund -> PAUSE (needs approval)
#   send_email     -> PAUSE (needs approval)
#   delete         -> PAUSE (needs approval)
#
# 3 actions paused for a human.
#   human APPROVES process_refund -> executed
#   human REJECTS  delete          -> discarded
# Read-only actions never waited; only the irreversible ones needed a person.

**Explanation**

A pure routing function, not a model call: the 'agent' is scripted so the gate itself is what you study. The whole idea is that the boundary between auto and pause lives in one editable set, and every proposed action passes through it first.

**How the code works - step by step**
- **`REQUIRES_APPROVAL`** - a set of tool names (`send_email`, `process_refund`, `delete`, `purchase`, `book_course`); this set is the safety control, expressed as data.
- **`gate(action)`** - returns `PAUSE` if the tool is in that set, `AUTO` otherwise.
- The loop over four `proposed` actions prints each verdict and collects the paused ones into `pending`.
- The final block scripts the human resolving the paused set - approve the refund, reject the delete - to show only irreversible actions ever waited.

**In one line:** classify by risk -> auto-run read-only, pause the irreversible.

**What you'll see:** `search` prints AUTO; `process_refund`, `send_email`, and `delete` each print PAUSE; then '3 actions paused for a human', the refund approved and the delete rejected.

## 2 - Route by confidence

The approval gate routes by an action's risk; this routes by the agent's certainty. Above a threshold it answers alone, below it escalates to a human. The trap is the confidence SIGNAL: raw token logprobs are systematically overconfident, so this cell uses ensemble agreement (the fraction of sampled answers that agreed) instead - a far more honest stand-in.

In [ ]:
# CONFIDENCE ROUTING - auto-act when sure, escalate when not. The agent rates its own
# confidence; above the threshold it answers, below it hands off to a human. The catch:
# raw logprobs are OVERCONFIDENT, so a robust signal uses ensemble agreement instead.

THRESHOLD = 0.80

# Scripted (query, ensemble_agreement) - the fraction of sampled answers that agreed.
# High agreement across samples = high real confidence (more honest than a raw logprob).
queries = [
    ("How much does the course cost?",              1.00),   # all 5 samples agree
    ("Can I use the certificate for a US visa?",    0.40),   # samples disagree -> unsure
    ("What is the refund policy?",                  0.90),   # strong agreement
    ("Will this replace my data-science job?",      0.20),   # opinion -> low agreement
]

def route(agreement):
    return "AUTO-ANSWER" if agreement >= THRESHOLD else "ESCALATE to a human"

print(f"Route each query by ensemble agreement (threshold {THRESHOLD}):")
escalated = 0
for q, agreement in queries:
    action = route(agreement)
    if action.startswith("ESCALATE"):
        escalated += 1
    print(f"  [{agreement:.2f}] {q[:42]:42} -> {action}")
print(f"\n{4 - escalated} answered automatically, {escalated} escalated.")
print("Note: a raw logprob would rate ALL of these ~0.9+ (overconfident); ensemble agreement")
print("separates the sure from the unsure. In a chain, uncertainty compounds: 3 x 0.9 = 0.73.")
# Output:
# Route each query by ensemble agreement (threshold 0.8):
#   [1.00] How much does the course cost?             -> AUTO-ANSWER
#   [0.40] Can I use the certificate for a US visa?   -> ESCALATE to a human
#   [0.90] What is the refund policy?                 -> AUTO-ANSWER
#   [0.20] Will this replace my data-science job?     -> ESCALATE to a human
#
# 2 answered automatically, 2 escalated.
# Note: a raw logprob would rate ALL of these ~0.9+ (overconfident); ensemble agreement
# separates the sure from the unsure. In a chain, uncertainty compounds: 3 x 0.9 = 0.73.

**Explanation**

A threshold comparator over scripted agreement scores, not a real sampling run. The point it drives home is in the closing note: a raw logprob would rate all four queries high, whereas ensemble agreement actually separates the sure from the unsure, and uncertainty compounds down a chain.

**How the code works - step by step**
- **`THRESHOLD = 0.80`** - the auto-vs-escalate cutoff.
- **`queries`** - four (question, agreement) pairs, from 1.00 (all samples agree) down to 0.20 (opinion, low agreement).
- **`route(agreement)`** - returns `AUTO-ANSWER` at or above the threshold, else `ESCALATE to a human`.
- The loop counts escalations and prints each verdict; the trailing note explains why logprobs would have failed here and that 3 x 0.9 chains to 0.73.

**In one line:** high ensemble agreement -> answer; low -> escalate.

**What you'll see:** The cost query (1.00) and refund policy (0.90) print AUTO-ANSWER; the visa (0.40) and job-replacement (0.20) print ESCALATE; then '2 answered automatically, 2 escalated' plus the logprob/compounding note.

## 3 - The interrupt/resume re-run trap

Approval gates and confidence routing decide WHEN to pause; this is HOW a production pause works - and it hides the most common HITL bug. In LangGraph, `interrupt()` suspends the graph and `Command(resume=value)` resumes it by re-running the ENTIRE node from the top, with `interrupt()` returning the value on that second pass. So any side effect before the interrupt fires twice unless it is idempotent.

In [ ]:
# THE INTERRUPT / RESUME TRAP - a durable HITL pause re-runs the WHOLE node on resume.
# LangGraph interrupt(payload) suspends; Command(resume=value) resumes by RE-RUNNING the
# node from the top, and interrupt() returns the value on that second pass. So a side effect
# BEFORE the interrupt runs TWICE unless it is idempotent (the 11.9 rule, applied to HITL).

# --- BUGGY node: a side effect before interrupt() runs on BOTH passes ---
emails_buggy = []
def node_buggy(resume=None):
    emails_buggy.append("welcome email")          # NOT idempotent - fires every pass
    if resume is None:
        return "INTERRUPT: awaiting approval"      # pass 1: suspend
    return f"executed (approved={resume})"         # pass 2: interrupt() returned the value

node_buggy(resume=None)                            # first run -> suspend
node_buggy(resume=True)                            # Command(resume=True) -> re-runs the node
print("BUGGY node: emails sent =", len(emails_buggy), "(the welcome email went out TWICE)")

# --- SAFE node: guard the side effect with an idempotency key ---
emails_safe, seen = [], set()
def node_safe(run_id, resume=None):
    if run_id not in seen:                         # idempotency guard (from 11.9)
        seen.add(run_id); emails_safe.append("welcome email")
    if resume is None:
        return "INTERRUPT: awaiting approval"
    return f"executed (approved={resume})"

node_safe("run-1", resume=None)
node_safe("run-1", resume=True)
print("SAFE  node: emails sent =", len(emails_safe), "(guarded, so it fired ONCE)")
print("Rule: everything before interrupt() must be idempotent, because the node re-runs on resume.")
# Output:
# BUGGY node: emails sent = 2 (the welcome email went out TWICE)
# SAFE  node: emails sent = 1 (guarded, so it fired ONCE)
# Rule: everything before interrupt() must be idempotent, because the node re-runs on resume.

**Explanation**

A minimal simulation of the re-run mechanic using two plain functions - no LangGraph needed. It contrasts a buggy node (side effect fires on every pass) with a guarded node (idempotency key from Lesson 11.9), so you see the double-fire and its fix side by side.

**How the code works - step by step**
- **`node_buggy`** - appends a 'welcome email' before the interrupt with no guard; called once to suspend and once to resume, so it runs twice.
- Printing `len(emails_buggy)` shows the email went out **twice** because the whole node re-ran.
- **`node_safe`** - guards the same append with a `seen` set keyed by `run_id`, the 11.9 idempotency pattern.
- Calling it with the same `run-1` on both passes leaves `emails_safe` at length **1**.

**In one line:** the node re-runs on resume, so everything before `interrupt()` must be idempotent.

**What you'll see:** 'BUGGY node: emails sent = 2', 'SAFE node: emails sent = 1', and the rule that pre-interrupt work must be idempotent because the node re-runs.

## 4 - Escalate to the right human, with context

Escalation is what happens after the agent decides to hand off. A flat 'send it to a human' wastes experts on easy cases and starves the hard ones. Production escalation is tiered - AI, L1 reviewer, L2 expert, kill switch - and just as important, every handoff carries a context bundle (history, confidence, suggested action) so the reviewer starts at line ten, not line one.

In [ ]:
# ESCALATION - route to the right human, WITH context. A tiered system: the AI handles
# what it can, an L1 reviewer takes low-confidence/low-risk, an L2 expert takes the hard
# cases, and a kill switch stops everything on a critical failure. Each handoff carries a
# context bundle so the human starts informed, not blind.

def route(risk, confidence, critical=False):
    if critical:                       return "KILL SWITCH (halt all + notify leadership)"
    if risk == "high":                 return "L2 EXPERT (domain decision)"
    if confidence < 0.80:              return "L1 REVIEWER (low confidence)"
    return "AI (auto-handle)"

def context_bundle(case):
    # Pass history + score + a suggested action so the reviewer starts at line 10, not line 1.
    return {"history": case["history"], "confidence": case["confidence"],
            "suggested": case["suggested"]}

cases = [
    {"id": 1, "risk": "low",  "confidence": 0.95, "critical": False, "history": "...", "suggested": "answer"},
    {"id": 2, "risk": "low",  "confidence": 0.55, "critical": False, "history": "...", "suggested": "clarify"},
    {"id": 3, "risk": "high", "confidence": 0.90, "critical": False, "history": "...", "suggested": "refund?"},
    {"id": 4, "risk": "high", "confidence": 0.99, "critical": True,  "history": "...", "suggested": "STOP"},
]
print("Route each case to the right tier, with context:")
for c in cases:
    tier = route(c["risk"], c["confidence"], c["critical"])
    bundle = context_bundle(c)
    print(f"  case {c['id']}: risk={c['risk']:4} conf={c['confidence']:.2f} -> {tier}")
print("\nEach escalation carries {history, confidence, suggested} so the human is not starting blind.")
# Output:
# Route each case to the right tier, with context:
#   case 1: risk=low  conf=0.95 -> AI (auto-handle)
#   case 2: risk=low  conf=0.55 -> L1 REVIEWER (low confidence)
#   case 3: risk=high conf=0.90 -> L2 EXPERT (domain decision)
#   case 4: risk=high conf=0.99 -> KILL SWITCH (halt all + notify leadership)
#
# Each escalation carries {history, confidence, suggested} so the human is not starting blind.

**Explanation**

Two small functions: a priority-ordered router and a bundle builder. It is decision logic, not a model call - the ordering of the checks is the lesson, because the critical flag must win before any risk or confidence test.

**How the code works - step by step**
- **`route(risk, confidence, critical)`** - checks in priority order: `critical` trips the KILL SWITCH first, then `high` risk goes to L2, then confidence below 0.80 goes to L1, else the AI auto-handles.
- **`context_bundle(case)`** - packages `history`, `confidence`, and a `suggested` action for each escalation.
- **`cases`** - four cases spanning low/high risk and a critical flag.
- The loop prints each case's tier; the closing line stresses every escalation carries its bundle.

**In one line:** route by critical -> risk -> confidence, and always hand over context.

**What you'll see:** Case 1 -> AI (auto-handle), case 2 -> L1 REVIEWER, case 3 -> L2 EXPERT, case 4 -> KILL SWITCH, and a note that each escalation carries {history, confidence, suggested}.

## 5 - Escalation fatigue: the silent killer

The failure mode that kills more HITL systems than missed approvals: 'more human review is safer' is wrong past a point. A reviewer drowning in a thousand approvals a day stops reading and clicks 'approve' - and now a human has formally signed off on mistakes nobody looked at. The effective catch rate is not constant; it FALLS as the escalation rate rises, and the sweet spot is roughly 10-15 percent.

In [ ]:
# ESCALATION FATIGUE - the SILENT KILLER of HITL. Too many escalations flood the reviewer,
# who then rubber-stamps, and the safety net collapses. Effective catch rate is NOT constant:
# it falls as the escalation rate rises. The practitioner sweet spot is roughly 10-15%.

def effective_catch_rate(escalation_rate):
    # A sharp reviewer catches ~95% of real problems; a flooded one rubber-stamps.
    # Attention degrades as the queue grows past a sustainable load.
    if escalation_rate <= 0.15:   attention = 0.95      # sustainable: reviewers stay sharp
    elif escalation_rate <= 0.30: attention = 0.80      # manageable but slipping
    elif escalation_rate <= 0.45: attention = 0.60
    else:                         attention = 0.40      # flooded -> rubber-stamping
    return attention

print("How the escalation rate changes the effective catch rate:")
for rate in [0.10, 0.20, 0.35, 0.60]:
    catch = effective_catch_rate(rate)
    tag = "sweet spot" if rate <= 0.15 else ("manageable" if rate <= 0.30 else "FATIGUE")
    print(f"  escalate {int(rate*100):>3}%  -> reviewer catches ~{int(catch*100)}% of real issues   ({tag})")
print("\nMore escalation is NOT safer past ~15%: flooded reviewers rubber-stamp and the net collapses.")
print("Escalate only the high-impact/uncertain cases; gather context first; keep the rate near 10-15%.")
# Output:
# How the escalation rate changes the effective catch rate:
#   escalate  10%  -> reviewer catches ~95% of real issues   (sweet spot)
#   escalate  20%  -> reviewer catches ~80% of real issues   (manageable)
#   escalate  35%  -> reviewer catches ~60% of real issues   (FATIGUE)
#   escalate  60%  -> reviewer catches ~40% of real issues   (FATIGUE)
#
# More escalation is NOT safer past ~15%: flooded reviewers rubber-stamp and the net collapses.
# Escalate only the high-impact/uncertain cases; gather context first; keep the rate near 10-15%.

**Explanation**

A tiny step model of reviewer attention as a function of load - a measurement harness, not a model call. It exists to make the counter-intuitive shape visible: pushing more work to humans past ~15 percent lowers the catch rate instead of raising it.

**How the code works - step by step**
- **`effective_catch_rate(escalation_rate)`** - maps the escalation rate to reviewer attention in bands: <=0.15 -> 0.95, <=0.30 -> 0.80, <=0.45 -> 0.60, else 0.40.
- The loop runs four rates (0.10, 0.20, 0.35, 0.60), tags each as sweet spot / manageable / FATIGUE, and prints the resulting catch rate.
- The closing lines state the rule: more escalation is not safer past ~15 percent; escalate only high-impact cases, with context, near 10-15 percent.

**In one line:** attention degrades with load, so past ~15 percent more review means less safety.

**What you'll see:** 10% -> ~95% caught (sweet spot), 20% -> ~80% (manageable), 35% -> ~60% (FATIGUE), 60% -> ~40% (FATIGUE), then the warning that flooded reviewers rubber-stamp.

## 6 - Progressive autonomy: earn trust, revoke it instantly

You do not hand a new agent the keys on day one, nor lock it at full-review forever. Progressive autonomy is the graduated middle: the agent earns trust through Audit -> Assist -> Automate, but only when a hard evidence gate passes, and always reversibly - a single critical error snaps the tier straight back to Audit with no code change.

In [ ]:
# PROGRESSIVE AUTONOMY - earn trust in stages, revoke it instantly. Start at full review
# (AUDIT), advance to mostly-auto (ASSIST), then sampling-only (AUTOMATE) - but only when
# hard gates pass, and ALWAYS reversibly: a critical error snaps the tier straight back.

TIERS = ["AUDIT", "ASSIST", "AUTOMATE"]           # 100% review -> risky-only -> sampling-only

def next_tier(current, accuracy, override_rate, critical_error):
    if critical_error:
        return "AUDIT"                             # instant rollback to full review
    i = TIERS.index(current)
    passes = accuracy >= 0.98 and override_rate < 0.02   # the advancement gate
    if passes and i < len(TIERS) - 1:
        return TIERS[i + 1]                        # advance one tier
    return current                                 # hold

# A stream of monthly review windows (accuracy, override_rate, critical_error):
windows = [
    (0.985, 0.010, False),   # good -> advance
    (0.990, 0.008, False),   # good -> advance
    (0.999, 0.001, True),    # a critical error -> rollback
    (0.985, 0.015, False),   # good -> advance
]
tier = "AUDIT"
print("Trust advances on the gate (accuracy >=98%, override <2%) and rolls back on a critical error:")
for i, (acc, ov, crit) in enumerate(windows, 1):
    tier = next_tier(tier, acc, ov, crit)
    note = "CRITICAL ERROR -> rollback" if crit else "gate passed -> advance"
    print(f"  window {i}: acc={acc:.3f} override={ov:.3f} {'crit' if crit else '    '} -> {tier:8} ({note})")
print("\nAutonomy is earned incrementally and revoked instantly; the progression is always reversible.")
# Output:
# Trust advances on the gate (accuracy >=98%, override <2%) and rolls back on a critical error:
#   window 1: acc=0.985 override=0.010      -> ASSIST   (gate passed -> advance)
#   window 2: acc=0.990 override=0.008      -> AUTOMATE (gate passed -> advance)
#   window 3: acc=0.999 override=0.001 crit -> AUDIT    (CRITICAL ERROR -> rollback)
#   window 4: acc=0.985 override=0.015      -> ASSIST   (gate passed -> advance)
#
# Autonomy is earned incrementally and revoked instantly; the progression is always reversible.

**Explanation**

A small state machine (finite-state trust ladder) driven by a stream of review windows. The two rules it enforces are the whole point: advancement is gated on evidence, and rollback on a critical error is instant and unconditional.

**How the code works - step by step**
- **`TIERS`** - the ladder `AUDIT -> ASSIST -> AUTOMATE` (100% review -> risky-only -> sampling-only).
- **`next_tier(...)`** - returns `AUDIT` immediately on `critical_error`; otherwise advances one tier only if the gate passes (accuracy >= 0.98 AND override rate < 0.02); else holds.
- **`windows`** - four monthly windows, the third carrying a critical error despite otherwise-perfect metrics.
- The loop threads `tier` through the windows and prints each transition.

**In one line:** advance on the gate, roll back to Audit instantly on any critical error.

**What you'll see:** Window 1 -> ASSIST, window 2 -> AUTOMATE, window 3 (critical) -> AUDIT rollback, window 4 -> ASSIST, closing that autonomy is earned incrementally and revoked instantly.

## 7 - Pick the HITL primitive by the need

The judgment call: which primitive implements the pause. Crash-durable approval that must survive a restart -> Temporal signals. A framework-native pause inside an agent run -> OpenAI needsApproval or LangGraph interrupt(). A quick local check -> your own manual gate. And for high-risk AI, human oversight is not optional - EU AI Act Article 14 legally requires it, phasing in through August 2026.

In [ ]:
# THE FRAMEWORK LANDSCAPE - pick the HITL primitive by the need. Crash-durable approval
# (survives a restart) -> Temporal signals. A framework-native pause -> OpenAI needsApproval
# or LangGraph interrupt(). A quick local check -> a manual gate. And for high-risk AI, human
# oversight is LEGALLY REQUIRED (EU AI Act Article 14), not optional.

def choose(needs_crash_durability, in_a_framework, high_risk_regulated):
    primitive = ("Temporal durable signals" if needs_crash_durability
                 else "OpenAI needsApproval / LangGraph interrupt()" if in_a_framework
                 else "a manual approval gate (your own code)")
    mandate = " + EU AI Act Article 14 human-oversight REQUIRED" if high_risk_regulated else ""
    return primitive + mandate

cases = [
    ("8-hour approval that must survive a restart", dict(needs_crash_durability=True,  in_a_framework=True,  high_risk_regulated=False)),
    ("pause a tool call inside an agent run",       dict(needs_crash_durability=False, in_a_framework=True,  high_risk_regulated=False)),
    ("a quick script that gates one action",        dict(needs_crash_durability=False, in_a_framework=False, high_risk_regulated=False)),
    ("a credit-decisioning agent (regulated)",      dict(needs_crash_durability=True,  in_a_framework=True,  high_risk_regulated=True)),
]
print("Route each need to the right HITL primitive:")
for name, props in cases:
    print(f"  {name:44} -> {choose(**props)}")
print("\nDurable approval -> Temporal; framework pause -> needsApproval / interrupt; and high-risk AI")
print("MUST allow a human to monitor, intervene, and override (EU AI Act Article 14).")
# Output:
# Route each need to the right HITL primitive:
#   8-hour approval that must survive a restart  -> Temporal durable signals
#   pause a tool call inside an agent run        -> OpenAI needsApproval / LangGraph interrupt()
#   a quick script that gates one action         -> a manual approval gate (your own code)
#   a credit-decisioning agent (regulated)       -> Temporal durable signals + EU AI Act Article 14 human-oversight REQUIRED
#
# Durable approval -> Temporal; framework pause -> needsApproval / interrupt; and high-risk AI
# MUST allow a human to monitor, intervene, and override (EU AI Act Article 14).

**Explanation**

A decision function that maps three boolean needs to a primitive and appends a compliance mandate when the system is regulated. It is a routing table in code, encoding the framework-selection judgment plus the legal trigger.

**How the code works - step by step**
- **`choose(needs_crash_durability, in_a_framework, high_risk_regulated)`** - crash-durability -> Temporal signals; else in a framework -> OpenAI needsApproval / LangGraph interrupt(); else a manual gate.
- When `high_risk_regulated` is true it appends the EU AI Act Article 14 human-oversight requirement.
- **`cases`** - four needs from an 8-hour durable approval to a regulated credit-decisioning agent.
- The loop prints each need's chosen primitive and any legal mandate.

**In one line:** match the primitive to durability and framework, and add Article 14 when the domain is high-risk.

**What you'll see:** Durable approval -> Temporal signals; in-run pause -> OpenAI needsApproval / LangGraph interrupt(); quick script -> a manual gate; the regulated credit agent -> Temporal + EU AI Act Article 14 human-oversight REQUIRED.

## 8 - The framework HITL APIs (illustrative)

A concrete look at the two current 2026 framework APIs behind the picker above. The OpenAI block is fully commented out; the Anthropic `can_use_tool` callback is real Python. Neither is meant to be executed here - it is a reference for the exact call shapes.

In [ ]:
# FRAMEWORK HITL - the current 2026 APIs (illustrative minimal example).
# OpenAI Agents SDK: mark a tool needs_approval; the run PAUSES and returns pending
# interruptions you approve or reject. Anthropic Claude Agent SDK: a can_use_tool
# callback returns allow/deny per tool call, shaped by permission modes.

# --- OpenAI Agents SDK (Python) ---
# from agents import Agent, function_tool          # pip install openai-agents

# @function_tool(needs_approval=True)          # money-moving tool -> the run pauses here
# def process_refund(order_id: str, amount: int) -> str:
#     return f"refunded {amount} on {order_id}"

# result = await Runner.run(agent, "refund order 123")
# if result.interruptions:                   # the run stopped, awaiting a human
#     result.interruptions[0].approve()      # or .reject() - your UI decides
#     result = await Runner.run(agent, result.to_input_list())

# --- Anthropic Claude Agent SDK (Python) ---
async def can_use_tool(tool_name, tool_input, context):
    if tool_name in {"Bash", "process_refund", "delete"}:
        return {"behavior": "deny", "message": "needs human approval"}   # gate the risky ones
    return {"behavior": "allow", "updatedInput": tool_input}             # auto-allow read-only

# ClaudeSDKClient(options=ClaudeAgentOptions(can_use_tool=can_use_tool,
#     permission_mode="default"))  # or bypassPermissions / acceptEdits / plan / dontAsk
# A PreToolUse hook runs before EVERY step and its deny wins even under bypassPermissions.
# Output: (illustrative minimal example - needs `pip install openai-agents` / `claude-agent-sdk`.)
# needs_approval pauses the run for sign-off; can_use_tool denies the risky tools and allows read-only.

**Explanation**

A reference/illustrative cell: the OpenAI half is commented pseudocode and the Anthropic half defines a callback that is never invoked, so running it produces nothing. It shows how each SDK flags a tool for approval versus gating tools with an allow/deny policy.

**How the code works - step by step**
- **OpenAI Agents SDK (commented)** - `@function_tool(needs_approval=True)` makes the run pause and return `interruptions` you `.approve()` or `.reject()`, then re-run with the updated input list.
- **`can_use_tool(tool_name, tool_input, context)`** - Anthropic callback returning `{'behavior': 'deny', ...}` for risky tools (`Bash`, `process_refund`, `delete`) and `{'behavior': 'allow', 'updatedInput': ...}` for read-only ones.
- Comments note the permission modes (`bypassPermissions` / `acceptEdits` / `plan` / `dontAsk`) and that a `PreToolUse` hook's deny wins even under `bypassPermissions`.

**In one line:** OpenAI flags a tool to pause the run; Anthropic denies risky tools and allows read-only via a callback.

**What you'll see:** No printed output - the OpenAI code is commented and `can_use_tool` is only defined. The trailing comments summarize that needs_approval pauses for sign-off and can_use_tool denies risky tools while allowing read-only ones.

Across seven concepts you drew one line precisely: put a human on the irreversible (approval gate) and the uncertain (confidence router by ensemble agreement, not logprobs), pause durably with interrupt/resume while keeping every pre-interrupt side effect idempotent, escalate to the right tier with a context bundle, watch the escalation rate so reviewers never rubber-stamp, earn autonomy in reversible Audit -> Assist -> Automate stages, and pick the pause primitive (Temporal / framework / local gate) the need demands. Every pattern here is plain decision logic you can run keyless, which is why oversight is a design choice you build in, never a default you inherit. Next, Lesson 11.11 measures these same agents - the escalation rate and override rate you tuned here become the monitoring signals.